# Part 1: From Theory to Practice (Feedforward Networks)

Companion code for the blog post [From Theory to Practice](https://johndorem.us/blog/from-theory-to-practice/), part of the *3Blue1Brown Neural Networks* series.

This notebook implements the feedforward pass for a simple fully-connected network (7 inputs, one hidden layer of 6 neurons, 3 outputs), following the equations presented in 3Blue1Brown's ["But what is a neural network?"](https://www.youtube.com/watch?v=aircAruvnKk).

It builds the computation up in two ways:

1. **`forward_naive`**: a direct, scalar translation of the video's equation, with explicit loops over neurons and inputs.
2. **`feed_forward`**: the same computation rewritten with NumPy's `@` operator and `zip()`, dispatching to BLAS instead of looping in Python.

Both produce identical outputs, which the last cell verifies numerically. The network is randomly initialized and untrained, so the output here is just random numbers. The point of this notebook is to get the weight and bias matrices built and aligned correctly, so the forward pass runs without a shape-mismatch error. Training (gradient descent and backpropagation) is the subject of the next post.

In [20]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Callable, Sequence

import numpy as np

In [21]:
def sigmoid(z: np.ndarray) -> np.ndarray:
    """1 / (1 + exp(-z))"""
    return 1.0 / (1.0 + np.exp(-z))


def sigmoid_prime(z: np.ndarray) -> np.ndarray:
    """Derivative of sigmoid, expressed in terms of sigmoid itself."""
    s = sigmoid(z)
    return s * (1.0 - s)


def relu(z: np.ndarray) -> np.ndarray:
    """max(0, z)"""
    return np.maximum(0.0, z)


def relu_prime(z: np.ndarray) -> np.ndarray:
    """Derivative of relu (0 for z <= 0, 1 for z > 0)."""
    return (z > 0).astype(z.dtype)


ACTIVATIONS: dict[str, tuple[Callable[[np.ndarray], np.ndarray], Callable[[np.ndarray], np.ndarray]]] = {
    "sigmoid": (sigmoid, sigmoid_prime),
    "relu": (relu, relu_prime),
}

In [26]:
class NeuralNetwork:
    """A fully-connected feedforward network with randomly initialized weights/biases.

    layer_sizes[i+1] neurons receive input from layer_sizes[i] neurons, so
    weights[i] has shape (layer_sizes[i+1], layer_sizes[i]) and biases[i] has
    shape (layer_sizes[i+1], 1), matching a^(l+1) = activation(W @ a^(l) + b).
    """

    def __init__(
        self,
        layer_sizes: Sequence[int],
        activations: str | Sequence[str] = "sigmoid", #default to sigmoid
        weight_scale: float = 0.1,
        seed: int | None = None,
    ) -> None:
        self.layer_sizes = list(layer_sizes)
        self.num_layers = len(self.layer_sizes) - 1

        if isinstance(activations, str):
            activation_names = [activations] * self.num_layers
        else:
            activation_names = list(activations)
        self.activation_names = activation_names
        self.activation_fns = [ACTIVATIONS[name][0] for name in activation_names]
        self.activation_derivative_fns = [ACTIVATIONS[name][1] for name in activation_names]

        rng = np.random.default_rng(seed)
        self.weights = [
            rng.standard_normal((self.layer_sizes[i + 1], self.layer_sizes[i])) * weight_scale
            for i in range(self.num_layers)
        ]
        self.biases = [
            rng.standard_normal((self.layer_sizes[i + 1], 1)) * weight_scale
            for i in range(self.num_layers)
        ]

    def summary(self) -> str:
        """Human-readable per-layer shape/activation table."""
        lines = []
        for i, (W, b, name) in enumerate(zip(self.weights, self.biases, self.activation_names)):
            lines.append(f"Layer {i + 1}: W {W.shape}   b {b.shape}   {name}")
        return "\n".join(lines)

    def forward_naive(self, input_activations):
        activations = list(input_activations)
        for layer in range(self.num_layers):
            weights = self.weights[layer]
            biases = self.biases[layer]
            activation_fn = self.activation_fns[layer]
            num_outputs, num_inputs = weights.shape

            next_activations = []
            for output_neuron in range(num_outputs):
                weighted_sum = biases[output_neuron, 0]
                for input_neuron in range(num_inputs):
                    weighted_sum = weighted_sum + weights[output_neuron, input_neuron] * activations[input_neuron]
                next_activations.append(activation_fn(weighted_sum))
            activations = next_activations
        return np.array(activations)
    
    def feed_forward(self, input_activations):
        activation = np.asarray(input_activations).reshape(-1, 1)
        for weights, biases, activation_fn in zip(self.weights, self.biases, self.activation_fns):
            z = weights @ activation + biases  # matrix multiply, not elementwise *
            activation = activation_fn(z)
        return activation.flatten()

    

In [27]:
network = NeuralNetwork([7, 6, 3], activations="sigmoid", seed=42)
print(network.summary())

Layer 1: W (6, 7)   b (6, 1)   sigmoid
Layer 2: W (3, 6)   b (3, 1)   sigmoid


In [28]:
#First, create a list of 7 random inputs between 0 and 1.
input_demo = np.random.default_rng(42).uniform(0, 1, size=7)
print(input_demo)


[0.77395605 0.43887844 0.85859792 0.69736803 0.09417735 0.97562235
 0.7611397 ]


In [29]:
naive_output = network.forward_naive(input_demo)
pythonic_output = network.feed_forward(input_demo)
print("forward_naive:   ", naive_output)
print("feed_forward: ", pythonic_output)
print("match:", np.allclose(naive_output, pythonic_output))

forward_naive:    [0.50621747 0.48408544 0.52509513]
feed_forward:  [0.50621747 0.48408544 0.52509513]
match: True
